<a href="https://colab.research.google.com/github/joyangelai/RAG/blob/main/RAGDoc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG Document QA Pipeline (Retrieval- Augmented Generation)

Building a system that lets users upload complex multi page PDFs and chat with them using an LLM that only references the document facts to prevent hallucinations.

In [18]:
# Installation
!pip install -q langchain langchain-community chromadb openai pypdf tiktoken langchain-openai

In [19]:
!pip install -q sentence-transformers

In [20]:
# importing necesssary libraries
import os
from google.colab import files
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

In [21]:
# Step 1: Upload PDF file
print("Please upload your PDF file: ")

uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]




Please upload your PDF file: 


Saving bitcoin.pdf to bitcoin (1).pdf


In [22]:
# Step 2: Document ingestion
print(f"\n[1/3] Parsing {pdf_filename}.....")

loader = PyPDFLoader(pdf_filename)
docs = loader.load()



[1/3] Parsing bitcoin (1).pdf.....


In [23]:
from langchain_community.document_loaders.onedrive_file import CHUNK_SIZE
# Step 3: Strategic Text chunking (using an overlap so sentences cut off at the edge of a chunk retain context)

print("[2/3] Chunking text into contextual segments...")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 700,
    chunk_overlap = 150
)
chunks =  text_splitter.split_documents(docs)
print(f"Document successfully split into {len(chunks)} distinct chunks")


[2/3] Chunking text into contextual segments...
Document successfully split into 41 distinct chunks


In [24]:
# Step 4: Vector store & Embeddings Generation

print("[3/3] Generating Vector embeddings and indexing to Chromadb...")

embeddings = HuggingFaceEmbeddings(model_name= "all-MiniLM-L6-v2")
vector_stores = Chroma.from_documents(chunks,embeddings)

print("Vector database is live and fully indexed!!")

[3/3] Generating Vector embeddings and indexing to Chromadb...


/tmp/ipykernel_493/3987050695.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name= "all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector database is live and fully indexed!!


In [ ]:
# NOT DONE YET
import os
from langchain_community.llms import HuggingFaceHub
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# 1. Authenticate with your free Hugging Face token
# @markdown Enter your free Hugging Face Access Token below:
HF_TOKEN = "" # @param {type:"string"}
os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN

# 2. Initialize a high-quality free open-source LLM (Mistral 7B)
# This model is hosted by Hugging Face for free API inferences
llm = HuggingFaceHub(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    model_kwargs={"temperature": 0.1, "max_new_tokens": 500}
)

# 3. Connect our free vector database retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# 4. Craft a strict system prompt to restrict hallucinations
system_prompt = (
    "You are an advanced data science assistant.\n"
    "Use ONLY the following pieces of retrieved context to answer the question at the end.\n"
    "If you do not know the answer or if it's not explicitly found in the text, say: \n"
    "'I cannot find the answer within the provided document.' Do not make things up.\n\n"
    "Context:\n{context}"
)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# 5. Assemble the RAG Chain
question_answer_chain = create_stuff_documents_chain(llm, prompt_template)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# 6. Run Your Query
# @markdown Enter your document question here:
user_query = "What is the primary problem this document attempts to solve?" # @param {type:"string"}

if user_query:
    if not HF_TOKEN:
        print("❌ Please paste your Hugging Face Access Token into the form first!")
    else:
        print(f"🔍 Querying vector indexes and generating response...")
        response = rag_chain.invoke({"input": user_query})

        print("\n💡 ANSWER:")
        print(response["answer"])
